# Prepare clean paid-purchase history

## Goal

Convert the private sales export into one row per **customer + date + product**.

Only ordinary paid product sales (`ПРОДАЖА`) are retained. Gift rows (`ПОДАРОК`) are removed completely and do not affect labels, quantities, recency, reorder timing, or the product catalogue.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_dir = project_root / "data" / "raw"

xlsx_raw_file = list(raw_data_dir.glob("*.xlsx"))[0]

if not xlsx_raw_file.exists():
    raise FileNotFoundError(f"Raw data file not found in {raw_data_dir}")
else:
    input_path = xlsx_raw_file

output_path = project_root / "data" / "interim" / "cleaned_purchases.csv"

## 2. Load, rename, and normalize columns

Customer and product codes are loaded as strings so pandas cannot interpret identifiers as numbers. Dates remain datetimes, and quantity remains numeric.

In [ ]:
column_mapping = {
    "КлиентДляОплатыКод": "customer_id",
    "ТоварКод": "product_id",
    "Категория": "product_category",
    "БизнесЛиния": "business_line",
    "ДатаПродажи": "purchase_date",
    "Количество": "quantity",
    "Gen_ Bus_ Posting Group": "transaction_type",
    "Gen_ Prod_ Posting Group": "item_type",
}

identifier_dtypes = {
    "КлиентДляОплатыКод": "string",
    "ТоварКод": "string",
}

df = pd.read_excel(input_path, dtype=identifier_dtypes)

missing_columns = sorted(set(column_mapping) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected source columns: {missing_columns}")

df = df.rename(columns=column_mapping)

text_columns = [
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
    "transaction_type",
    "item_type",
]

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(f"Loaded {len(df):,} source rows.")
df.head(10)

## 3. Keep positive paid product purchases

A retained row must identify a customer, date, and product; represent an actual product (`ТОВАР`); be an ordinary sale (`ПРОДАЖА`); and have a positive quantity. Gift rows and every other transaction type are excluded before aggregation.

In [ ]:
required_text_columns = [
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
]
required_text = df[required_text_columns]
complete_required_values = (
    required_text.notna().all(axis=1)
    & required_text.ne("").all(axis=1)
    & df["purchase_date"].notna()
    & df["quantity"].notna()
)

purchase_mask = (
    complete_required_values
    & df["item_type"].eq("ТОВАР")
    & df["transaction_type"].eq("ПРОДАЖА")
    & df["quantity"].gt(0)
    & df["product_id"].str.startswith('ТОВ', na=False)
)

product_purchases = df.loc[purchase_mask].copy()

gift_product_rows = int(
    (
        df["item_type"].eq("ТОВАР")
        & df["transaction_type"].eq("ПОДАРОК")
        & df["quantity"].gt(0)
        & df["product_id"].str.startswith('ТОВ', na=True)
    ).sum()
)
assert product_purchases["transaction_type"].eq("ПРОДАЖА").all()
print(f"Excluded {gift_product_rows:,} positive gift-product rows.")
product_purchases.head(10)

## 4. Combine duplicate customer-date-product rows

The grouping key is `customer_id + purchase_date + product_id`. Different products on the same date remain separate. Repeated rows for the same product are combined.

Rows missing any required identifier or product classification are removed before aggregation. Purchase quantities are summed, and `first` retains the classification attached to the earliest retained source row.

In [ ]:
grouping_columns = ["customer_id", "purchase_date", "product_id"]

cleaned_purchases = (
    product_purchases
    .groupby(
        grouping_columns,
        as_index=False,
        sort=False,
    )
    .agg(
        quantity=("quantity", "sum"),
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
    )
    .sort_values(grouping_columns)
    .reset_index(drop=True)
)

cleaned_purchases.head(10)

## 5. Validate and save

The checks stop the notebook if duplicate keys, non-positive purchases, gift-related fields, or missing numeric values survive. The validated table is then saved as CSV with dates written as `YYYY-MM-DD`.

In [ ]:
duplicate_count = int(cleaned_purchases.duplicated(grouping_columns).sum())
non_positive_purchase_count = int(cleaned_purchases["quantity"].le(0).sum())

assert duplicate_count == 0, f"Found {duplicate_count} duplicate grouping keys."
assert non_positive_purchase_count == 0, (
    f"Found {non_positive_purchase_count} non-positive purchases."
)
assert cleaned_purchases.notna().all().all()
assert cleaned_purchases[required_text_columns].ne("").all().all()
assert {
    "gift_quantity",
    "received_quantity",
    "paid_quantity",
}.isdisjoint(cleaned_purchases.columns)

output_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_purchases.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_purchases = pd.read_csv(
    output_path,
    dtype={
        "customer_id": "string",
        "product_id": "string",
        "product_category": "string",
        "business_line": "string",
    },
    parse_dates=["purchase_date"],
)
assert len(saved_purchases) == len(cleaned_purchases)
assert list(saved_purchases.columns) == list(cleaned_purchases.columns)

validation_summary = pd.Series({
    "source_rows": len(df),
    "eligible_paid_purchase_rows": len(product_purchases),
    "excluded_positive_gift_rows": gift_product_rows,
    "cleaned_customer_date_product_rows": len(cleaned_purchases),
    "duplicate_source_rows_combined": len(product_purchases) - len(cleaned_purchases),
    "remaining_duplicate_keys": duplicate_count,
    "customers": cleaned_purchases["customer_id"].nunique(),
    "products": cleaned_purchases["product_id"].nunique(),
})

print(f"Saved validated data to {output_path}")
validation_summary